# E08 · 01 单篇 PDF → 完整 MinerU 产出（md + 图 + block + 溯源）

输入：`data/raw/e08/pdf/*.pdf`。单篇看效果时输出 `_scratch/ocr/<主名>/`，整目录批量跑
（最后一个 cell）输出 `data/raw/e08/md/<主名>/`。两者都是探索产物，**不是**
`data/processed/` 的流水线工件。

    <stem>.md          正文：caption 在图旁、模型读图的描述收在 <details> 里、
                       标题带 #、公式用 $ / $$、页眉页脚页码不在正文
    blocks.json        服务返回的逐页 block 原样（不做任何改写）+ 裁图清单
    content_list.json  装配后的扁平清单（官方 content_list 形态）：img_path、
                       content、*_caption、*_footnote、sub_type、sub_images、bbox
    images/            每个视觉主体一张裁图；多图容器裁整图，不裁碎片
    pages/             整页渲染，默认不留（keep_page_renders=True 才有）
    meta.json          溯源：pdf sha256、页数、dpi、服务地址、模型、库版本、耗时、计数

**口径声明**

- OCR 结果是**机器抽取的文本，不是事实**：表格、公式、图注的还原都可能出错，
  任何要进入论断的数字都应回 PDF 原文核对。同一篇重跑两次不保证逐字相同。
- `<details>` 里的内容是**模型看图写的**（`image_analysis`，默认开）：flowchart → mermaid、
  chart → markdown 表、pure_table → HTML、natural_image → caption。实测 mermaid 会
  编造节点、chart 转成的表是读图估的数 —— **不可当实验数据引用**，要用就回裁图核对。
- 图的像素是按 block 的归一化 bbox 自己从页图裁的（`mineru_vl_utils` 不写任何文件）。
  `bbox` 与 dpi 无关，所以 `crop_dpi` 可以高于 OCR 的 `dpi`。
- block → md / content_list 的装配逻辑在 `e08.mineru_assemble`，移植自官方 mineru
  （`references/repos/MinerU` @ 3.4.0），偏离之处写在该模块 docstring 里。
- 服务地址取自 `experiments/e08/.env` 的 `MINERU_SERVICE=`（或同名环境变量，或
  `server_url=` 参数；优先级：参数 > 环境变量 > `.env`）。找不到就抛 `RuntimeError`，
  不会悄悄回落到 localhost —— 所以产物属于哪个服务是确定的。
- 渲染依赖系统二进制 `mutool`（Debian/Ubuntu：`apt install mupdf-tools`）。
- 本 notebook 是**探索面**。若这批产出成为文档引用数字的来源，按 AGENTS.md
  先提升为 `src/e08/pN_*.py`，产物写进 `data/processed/e08/`。

In [ ]:
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "AGENTS.md").exists():          # notebook 可能从任意目录启动
    REPO = REPO.parent

from e08.mineru_ocr import aocr_bundle

PDF_DIR = REPO / "data/raw/e08/pdf"
OUT_ROOT = Path.cwd() / "_scratch/ocr"            # 探索产物，不进 data/processed

pdfs = sorted(PDF_DIR.glob("*.pdf"))
print(f"{len(pdfs)} 篇：")
for p in pdfs[:5]:
    print(" ", p.stem[:70])
print("  ...")

pdf = pdfs[0]                                     # 换成你要看的那篇
assert pdf.is_file(), pdf
pdf.stem

In [ ]:
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "AGENTS.md").exists():          # notebook 可能从任意目录启动
    REPO = REPO.parent

from e08.mineru_ocr import aocr_bundle

PDF_DIR = REPO / "data/raw/e08/pdf"
OUT_ROOT = Path.cwd() / "_scratch/ocr"            # 探索产物，不进 data/processed

pdfs = sorted(PDF_DIR.glob("*.pdf"))
print(f"{len(pdfs)} 篇：")
for p in pdfs[:5]:
    print(" ", p.stem[:70])
print("  ...")

pdf = pdfs[0]                                     # 换成你要看的那篇
assert pdf.is_file(), pdf
pdf.stem

## 跑一篇

单独一个 cell：一篇 20 页上下的论文要几十秒到几分钟。同步版 `ocr_bundle` 在 Jupyter 里
会直接抛 `RuntimeError`（已有事件循环），所以用 `await`。

目录已存在且非空会抛 `FileExistsError` —— 重跑同一篇要 `overwrite=True`，免得手滑重烧一遍。

In [ ]:
bundle = await aocr_bundle(pdf, OUT_ROOT, crop_dpi=300, overwrite=True)
bundle.relative_to(REPO)

## 看产出

先看溯源和块计数，再看图,最后看 md 开头。这三样合起来能回答"这篇抽到了什么、图留下了几张"。

In [ ]:
import json

meta = json.loads((bundle / "meta.json").read_text(encoding="utf-8"))
c = meta["counts"]
print(f"{meta['page_count']} 页 | 原始 block {c['raw_blocks']} | {meta['run']['elapsed_s']}s "
      f"| dpi {meta['params']['dpi']}/{meta['params']['crop_dpi']}")
print("服务:", meta["service"]["server_url"], "来源:", meta["service"]["url_origin"])
print(f"视觉主体 {c['visual_total']}（其中多图容器 {c['visual_containers']}，"
      f"配上 caption {c['visual_with_caption']}）→ 裁图 {c['crops']} 张")
print("视觉:", c["visual"], "| 正文:", c["flow"], "| 分流掉的版面附属物:", c["discarded"])

In [ ]:
from IPython.display import Image as IPyImage, display

crops = sorted((bundle / "images").glob("*.png"))
print(f"{len(crops)} 张裁图：", [p.name for p in crops][:10])
for p in crops[:3]:                               # 只看前三张，别把 notebook 撑爆
    print(p.name)
    display(IPyImage(filename=str(p), width=520))

In [ ]:
md_text = (bundle / f"{pdf.stem}.md").read_text(encoding="utf-8")
content_list = json.loads((bundle / "content_list.json").read_text(encoding="utf-8"))
print(f"md {len(md_text)} chars | 图片引用 {md_text.count('![')} 处 | 标题 "
      f"{sum(1 for line in md_text.splitlines() if line.startswith('#'))} 个 | "
      f"content_list {len(content_list)} 条")
print(md_text[:1200])

## 批量：整目录

`data/raw/e08/pdf/*.pdf` 全部跑一遍，产出落在 `data/raw/e08/md/<主名>/`
（`data/raw/` 已在 `.gitignore` 里，不会进版本库）。

**串行**，不 `gather`：单篇内部已经按页并发 24 路打服务，20 篇再叠上去就是 480 个
并发请求，服务那边扛不住，出错也分不清是谁的。

已有非空产出的篇目**跳过**而不是覆盖 —— 这个 cell 可以中断后重跑接着做。要重烧某一篇，
删掉它的目录，或单独 `await aocr_bundle(pdf, MD_ROOT, crop_dpi=300, overwrite=True)`。
单篇失败只记下来继续下一篇，最后汇总；失败清单要逐个看，别默认"跑完了就是好的"。


In [ ]:
import json, time, traceback

MD_ROOT = REPO / "data/raw/e08/md"

done, skipped, failed = [], [], []
t0 = time.monotonic()
for i, p in enumerate(pdfs, 1):
    tag = f"[{i:2}/{len(pdfs)}]"
    target = MD_ROOT / p.stem
    if target.exists() and any(target.iterdir()):      # 断点续跑
        skipped.append(p.stem)
        print(f"{tag} skip  {p.stem[:56]}")
        continue
    try:
        bundle_i = await aocr_bundle(p, MD_ROOT, crop_dpi=300)
    except Exception as exc:                            # 一篇挂掉不拖累整批
        failed.append((p.stem, f"{type(exc).__name__}: {exc}"))
        print(f"{tag} FAIL  {p.stem[:56]}\n        {failed[-1][1]}")
        traceback.print_exc(limit=1)
        continue
    m = json.loads((bundle_i / "meta.json").read_text(encoding="utf-8"))
    c = m["counts"]
    done.append(p.stem)
    print(f"{tag} ok    {m['page_count']:>3}p block {c['raw_blocks']:>4} "
          f"视觉 {c['visual_total']:>2}(配 caption {c['visual_with_caption']:>2}) "
          f"裁图 {c['crops']:>2} {m['run']['elapsed_s']:>6.1f}s  {p.stem[:44]}")

print(f"\n{len(done)} 成功 / {len(skipped)} 跳过 / {len(failed)} 失败"
      f"，共 {time.monotonic() - t0:.0f}s → {MD_ROOT.relative_to(REPO)}")
for stem, err in failed:
    print("  FAIL", stem, "|", err)
